# RobustLeaf-Net: run every experiment and build the paper
Runtime -> Change runtime type -> **GPU (T4)**. Run the cells in order.
Expected time on a free T4: several hours for the full 6 variants x 3 seeds. Use the `EXTRA` reduced setting in step 4 for a faster run, and **state any reduction in the paper**.

In [ ]:
# 1. Get the project: upload RobustLeaf-Net.zip, or clone your GitHub repo
from google.colab import files
up = files.upload()          # choose RobustLeaf-Net.zip
!unzip -qo RobustLeaf-Net.zip
# alternative: !git clone https://github.com/YOUR_USERNAME/RobustLeaf-Net.git
%cd RobustLeaf-Net
!pip -q install -r requirements.txt

## 2. Data
Upload your Kaggle API key (kaggle.com -> Settings -> Create New Token) to download PlantVillage.
Field images: PlantDoc (public) below, or put your own photos in folders named like the PlantVillage classes.

In [ ]:
files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p data --unzip
!ls data
!ln -sfn "$(ls -d data/*/color | head -1)" pv_color
!ls pv_color | head -40

In [ ]:
# Field images (PlantDoc). Check the folder names printed below; edit the mapping if they differ.
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset field_src
!ls field_src/test

In [ ]:
import json
# PlantDoc folder name -> PlantVillage class name (only classes shared by both datasets).
mapping = {
 "Apple Scab Leaf": "Apple___Apple_scab", "Apple rust leaf": "Apple___Cedar_apple_rust",
 "Apple leaf": "Apple___healthy", "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
 "Bell_pepper leaf": "Pepper,_bell___healthy", "Blueberry leaf": "Blueberry___healthy",
 "Cherry leaf": "Cherry_(including_sour)___healthy", "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
 "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight", "Corn rust leaf": "Corn_(maize)___Common_rust_",
 "Peach leaf": "Peach___healthy", "Potato leaf early blight": "Potato___Early_blight",
 "Potato leaf late blight": "Potato___Late_blight", "Raspberry leaf": "Raspberry___healthy",
 "Soyabean leaf": "Soybean___healthy", "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
 "Strawberry leaf": "Strawberry___healthy", "Tomato Early blight leaf": "Tomato___Early_blight",
 "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot", "Tomato leaf": "Tomato___healthy",
 "Tomato leaf bacterial spot": "Tomato___Bacterial_spot", "Tomato leaf late blight": "Tomato___Late_blight",
 "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus", "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
 "Tomato mold leaf": "Tomato___Leaf_Mold", "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
 "grape leaf": "Grape___healthy", "grape leaf black rot": "Grape___Black_rot",
}
json.dump(mapping, open("mapping.json", "w"))
# how many field images match a PlantVillage class?
import os
pv = set(os.listdir("pv_color"))
ok = {k: v for k, v in mapping.items() if v in pv and os.path.isdir(f"field_src/test/{k}")}
n = sum(len(os.listdir(f"field_src/test/{k}")) for k in ok)
print("shared classes:", len(ok), "field images:", n)

## 3. Smoke test (2 minutes) - make sure everything runs before the long jobs

In [ ]:
!python make_synthetic.py synthetic_data
!python train.py --data_dir synthetic_data --run_name smoke --model transfer --cbam --sbr --attn_sup --hier --no_pretrained --epochs_head 1 --epochs_ft 1 --batch 8 --split_file runs/smoke_splits.json

## 4. Run the six variants (A0-A5) for three seeds
Full setting: leave `EXTRA` empty. Reduced setting (faster): `EXTRA="--max_per_class 300 --epochs_head 4 --epochs_ft 4"`.
If you change `EXTRA`, delete `runs/splits.json` first.

In [ ]:
import os
os.environ["EXTRA"] = ""        # e.g. "--max_per_class 300 --epochs_head 4 --epochs_ft 4"
os.environ["OODMAP"] = "mapping.json"
!rm -f runs/splits.json
for seed in (42, 1, 2):
    os.environ["SEED"] = str(seed)
    !bash run_ablation.sh pv_color field_src/test

## 5. Figures, numbers and the paper

In [ ]:
!python make_paper_figures.py --data_dir pv_color --field_dir field_src/test
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# Fill the numbers: use the counts printed in step 2 and your hardware names
!python fill_paper.py --field_images {n} --field_classes {len(ok)} --gpu "NVIDIA T4 (Google Colab)" --device "Colab CPU"

In [ ]:
!python make_results_table.py
!zip -qr paper.zip paper results_table.md results_table.csv runs/*/metrics.json runs/*/classification_report.txt
files.download("paper.zip")   # upload the paper/ folder to Overleaf and compile with pdfLaTeX